# 03 - QLoRA Fine-Tuning

Fine-tune Qwen2.5-7B-Instruct with QLoRA using Unsloth.

**Prerequisites:** Run `02_data_preprocessing.ipynb` first to generate `data/splits/`.

**Runtime:** Kaggle T4 GPU, ~1.5-2 hours for v1 baseline.

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA capability: {props.major}.{props.minor}")

assert props.major >= 7, f"GPU {gpu_name} is too old. Select T4 in notebook settings."
print("GPU check passed.")

In [ ]:
import os
os.chdir('/kaggle/working')

# Upload src/ as a Kaggle dataset and mount it, or clone from GitHub
if not os.path.exists('fingpt-qlora'):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir('fingpt-qlora')

# Run data pipeline if splits don't exist
if not os.path.exists('data/splits/train.json'):
    print("Running data pipeline...")
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

# Verify data splits exist
assert os.path.exists('data/splits/train.json'), "Data pipeline failed!"
print("Data splits found.")


## 1. Load Model

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

## 2. Apply LoRA Adapters

In [ ]:
# === EXPERIMENT CONFIG ===
# Change these for ablation experiments:
# v1 (baseline): r=16, lr=2e-4, 3 epochs
# v2 (high rank): r=32, lr=1e-4, 3 epochs  
# v3 (fewer epochs): r=16, lr=2e-4, 1 epoch
# v4 (low rank): r=8, lr=2e-4, 3 epochs

LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
EXPERIMENT_NAME = "v1_baseline"

print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 3. Prepare Dataset

In [ ]:
import json
from datasets import Dataset

def load_sharegpt_split(path: str) -> Dataset:
    with open(path) as f:
        data = json.load(f)
    
    texts = []
    for ex in data:
        messages = []
        for turn in ex['conversations']:
            role = turn['from']
            if role == 'gpt':
                role = 'assistant'
            messages.append({'role': role, 'content': turn['value']})
        
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append({'text': text})
    
    return Dataset.from_list(texts)

train_dataset = load_sharegpt_split('data/splits/train.json')
val_dataset = load_sharegpt_split('data/splits/val.json')

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")
print(f"\nSample text length: {len(train_dataset[0]['text'])} chars")
print(f"Sample preview: {train_dataset[0]['text'][:300]}...")

## 4. Training Configuration

In [ ]:
from trl import SFTTrainer, SFTConfig

BATCH_SIZE = 2
GRAD_ACCUM = 8
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
TOTAL_STEPS = (len(train_dataset) * NUM_EPOCHS) // EFFECTIVE_BATCH

print(f"Effective batch size: {EFFECTIVE_BATCH}")
print(f"Total training steps: {TOTAL_STEPS}")
print(f"Warmup steps (5%): {int(TOTAL_STEPS * 0.05)}")
print(f"Eval every 100 steps: {TOTAL_STEPS // 100} checkpoints")
print(f"\nEstimated time: ~{TOTAL_STEPS * 1.0 / 60:.0f} minutes (Unsloth ~1s/step on T4)")

In [ ]:
training_args = SFTConfig(
    output_dir=f"outputs/{EXPERIMENT_NAME}",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_steps=500,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=500,
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

print("Training config:")
print(f"  Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {EFFECTIVE_BATCH}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Scheduler: cosine")
print(f"  Optimizer: adamw_8bit")

## 5. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print("Starting training...")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"="*50)

trainer.train()

## 6. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

os.makedirs('results/figures', exist_ok=True)

log_history = trainer.state.log_history

train_loss = [(l['step'], l['loss']) for l in log_history if 'loss' in l]
eval_loss = [(l['step'], l['eval_loss']) for l in log_history if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(10, 5))
if train_loss:
    steps, losses = zip(*train_loss)
    ax.plot(steps, losses, label='Train Loss', alpha=0.7)
if eval_loss:
    steps, losses = zip(*eval_loss)
    ax.plot(steps, losses, label='Eval Loss', marker='o', markersize=4)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title(f'Training Curves - {EXPERIMENT_NAME}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'results/figures/training_curve_{EXPERIMENT_NAME}.png', dpi=150)
plt.show()
print(f"Saved: results/figures/training_curve_{EXPERIMENT_NAME}.png")

## 7. Generate Sample Outputs

In [ ]:
FastLanguageModel.for_inference(model)

SAMPLE_PROMPTS = [
    "Analyze the sentiment of this financial news: 'Tesla shares surged 12% after record Q3 deliveries'",
    "What are the key risks of investing in emerging market bonds?",
    "Summarize: Company X reported EPS of $2.50 vs $2.30 expected, revenue up 15% YoY.",
]

for i, prompt in enumerate(SAMPLE_PROMPTS):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Prompt {i+1}: {prompt[:80]}...")
    print(f"{'='*60}")
    print(response)

## 8. Save Model

In [ ]:
# Save LoRA adapter only (small, ~50MB)
lora_path = f"outputs/{EXPERIMENT_NAME}/lora_adapter"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"LoRA adapter saved to {lora_path}")

# Save merged model (full 16-bit, ~14GB)
merged_path = f"outputs/{EXPERIMENT_NAME}/merged"
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {merged_path}")

In [ ]:
# Optional: Push to Hugging Face Hub
# Uncomment and set your HF token:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
# model.push_to_hub_merged("W-Kaski/fingpt-qlora-qwen2.5-7b", tokenizer, save_method="merged_16bit")
# print("Pushed to Hugging Face Hub!")

In [ ]:
# Save training logs
import json
logs_path = f"results/training_logs/{EXPERIMENT_NAME}.json"
os.makedirs(os.path.dirname(logs_path), exist_ok=True)
with open(logs_path, 'w') as f:
    json.dump(log_history, f, indent=2)
print(f"Training logs saved to {logs_path}")

print(f"\n{'='*50}")
print(f"Experiment {EXPERIMENT_NAME} complete!")
print(f"{'='*50}")